## 1. 모듈 로드와 프로젝트 경로 설정

- 노트북을 어느 위치에서 열어도 `data_collection` 프로젝트 루트를 찾도록 설정합니다.
- `loader` 와 `scripts.vpn_switcher` 모듈을 다시 불러와서 최신 코드를 바로 반영합니다.
- 여기서는 상세페이지 크롤링과 VPN 전환 로직을 함께 준비합니다.


In [ ]:
import asyncio
import csv
import importlib
import inspect
import json
import sys
import time
from pathlib import Path


def find_project_dir(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")


PROJECT_DIR = find_project_dir(Path.cwd().resolve())
SRC_DIR = PROJECT_DIR / "src"

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import loader as loader_pkg
import loader.crawl as crawl_pkg
import loader.crawl.detail_page as detail_page_module
import scripts.vpn_switcher as vpn_switcher_module

detail_page_module = importlib.reload(detail_page_module)
crawl_pkg = importlib.reload(crawl_pkg)
loader_pkg = importlib.reload(loader_pkg)
vpn_switcher = importlib.reload(vpn_switcher_module)

BlockedRequestError = loader_pkg.BlockedRequestError
SlowResponseError = loader_pkg.SlowResponseError
StaleListingError = loader_pkg.StaleListingError
create_browser_context = loader_pkg.create_browser_context
crawl_detail_page = loader_pkg.crawl_detail_page

print("프로젝트 경로:", PROJECT_DIR)
print("crawl_detail_page 시그니처:", inspect.signature(crawl_detail_page))
print("vpn_switcher 모듈:", vpn_switcher.__file__)


## 2. 입력 경로와 실행 옵션 설정

- PDF 크롤링 때 사용한 것과 같은 입력 CSV(`search_scope.csv`)를 그대로 사용합니다.
- 결과물은 JSON, CSV, 로그 파일로 `output/shallow_only` 아래에 저장합니다.
- 한 번에 25개씩 배치로 처리하고, 차단이 감지되면 한국 VPN 서버만 순환합니다.


In [ ]:
CSV_PATH = PROJECT_DIR / "raw" / "page_urls" / "search_scope.csv"
OUTPUT_DIR = SRC_DIR / "output" / "shallow_only"
CHECKPOINT_PATH = OUTPUT_DIR / "shallow_checkpoint.json"
RESULTS_JSON_PATH = OUTPUT_DIR / "detail_pages.json"
FAILED_JSON_PATH = OUTPUT_DIR / "failed_rows.json"
RESULTS_CSV_PATH = OUTPUT_DIR / "detail_pages.csv"
FAILED_CSV_PATH = OUTPUT_DIR / "failed_rows.csv"
LOG_PATH = OUTPUT_DIR / "shallow_only.log"

MAX_ITEMS = None
BATCH_SIZE = 25
DETAIL_CONCURRENCY = 3
LOG_EVERY = 25
DETAIL_TIMEOUT_MS = 10_000
BLOCKED_ERROR_THRESHOLD = 1
SLOW_RETRY_LIMIT = 2
SLOW_TO_BLOCK_THRESHOLD = 3
BATCH_SLOW_BLOCK_THRESHOLD = 2

VPN_PRIMARY_TARGETS = ["kr115", "kr142", "kr106", "kr117", "kr153", "kr94", "kr120"]
VPN_CONNECT_TIMEOUT_SECONDS = 45.0
VPN_READY_TIMEOUT_SECONDS = 90.0
VPN_STATUS_POLL_INTERVAL_SECONDS = 2.0
VPN_PROBE_URL = "https://www.carku.kr/search/car-detail.html?wDemoNo=0361001191"
VPN_PROBE_TIMEOUT_SECONDS = 15.0
VPN_MAX_PROBE_LATENCY_SECONDS = 6.0
VPN_ROTATION_MAX_ATTEMPTS = len(VPN_PRIMARY_TARGETS)
VPN_RESET_WAIT_SECONDS = 180.0

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH.touch(exist_ok=True)

print(f"입력 CSV 경로: {CSV_PATH}")
print(f"출력 디렉터리: {OUTPUT_DIR}")
print(f"체크포인트 경로: {CHECKPOINT_PATH}")
print(f"결과 JSON 경로: {RESULTS_JSON_PATH}")
print(f"실패 JSON 경로: {FAILED_JSON_PATH}")
print(f"결과 CSV 경로: {RESULTS_CSV_PATH}")
print(f"실패 CSV 경로: {FAILED_CSV_PATH}")
print(f"로그 경로: {LOG_PATH}")
print(f"배치 크기: {BATCH_SIZE}")
print(f"상세 페이지 동시 처리 개수: {DETAIL_CONCURRENCY}")
print(f"상세 페이지 타임아웃(ms): {DETAIL_TIMEOUT_MS}")
print(f"느린 응답 재시도 횟수: {SLOW_RETRY_LIMIT}")
print(f"VPN 1순위 서버: {VPN_PRIMARY_TARGETS}")
print(f"VPN 실패 후 대기 시간(초): {VPN_RESET_WAIT_SECONDS}")
print(f"최대 대상 개수: {MAX_ITEMS if MAX_ITEMS is not None else '전체'}")
print("차단 감지 시: 한국 서버만 순환 -> 전부 실패 시 3분 대기 -> 다시 처음 서버부터 재시도")


## 3. 체크포인트, CSV 저장, 로그, VPN 상태 유틸리티

- 중간에 멈춰도 이어서 실행할 수 있도록 체크포인트와 런타임 상태를 저장합니다.
- 성공/실패 결과를 JSON과 CSV로 모두 기록합니다.
- 한국 서버를 모두 시도했는데도 전환이 실패하면 3분 대기 후 처음 서버부터 다시 시작합니다.


In [ ]:
def log(message: str) -> None:
    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {message}"
    print(line)
    with LOG_PATH.open("a", encoding="utf-8") as handle:
        handle.write(line + "\n")


CSV_FIELDNAMES = [
    "status",
    "category",
    "order_index",
    "demo_no",
    "detail_url",
    "source_page",
    "registration_number",
    "title",
    "price_text",
    "price_manwon",
    "model_year_text",
    "first_registration_text",
    "fuel_type",
    "transmission",
    "color",
    "mileage_text",
    "mileage_km",
    "vin",
    "performance_number",
    "seller_name",
    "seller_phone",
    "dealer_name",
    "dealer_phone",
    "dealer_address",
    "elapsed_seconds",
    "error",
]


def flatten_result_for_csv(result: dict[str, object]) -> dict[str, object]:
    detail_page = result.get("detail_page") or {}
    return {
        "status": result.get("status"),
        "category": result.get("category"),
        "order_index": result.get("order_index"),
        "demo_no": result.get("demo_no"),
        "detail_url": result.get("detail_url"),
        "source_page": result.get("source_page"),
        "registration_number": result.get("registration_number") or detail_page.get("registration_number"),
        "title": detail_page.get("title"),
        "price_text": detail_page.get("price_text"),
        "price_manwon": detail_page.get("price_manwon"),
        "model_year_text": detail_page.get("model_year_text"),
        "first_registration_text": detail_page.get("first_registration_text"),
        "fuel_type": detail_page.get("fuel_type"),
        "transmission": detail_page.get("transmission"),
        "color": detail_page.get("color"),
        "mileage_text": detail_page.get("mileage_text"),
        "mileage_km": detail_page.get("mileage_km"),
        "vin": detail_page.get("vin"),
        "performance_number": detail_page.get("performance_number"),
        "seller_name": detail_page.get("seller_name"),
        "seller_phone": detail_page.get("seller_phone"),
        "dealer_name": detail_page.get("dealer_name"),
        "dealer_phone": detail_page.get("dealer_phone"),
        "dealer_address": detail_page.get("dealer_address"),
        "elapsed_seconds": result.get("elapsed_seconds"),
        "error": result.get("error"),
    }


def write_csv_rows(path: Path, rows: list[dict[str, object]]) -> None:
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=CSV_FIELDNAMES)
        writer.writeheader()
        for row in rows:
            writer.writerow(flatten_result_for_csv(row))


def build_default_runtime_state() -> dict[str, object]:
    return {
        "vpn_target_index": 0,
        "vpn_switch_history": [],
        "last_vpn_result": None,
        "last_public_ip": None,
    }


def normalize_runtime_state(runtime_state: dict[str, object] | None) -> dict[str, object]:
    payload = build_default_runtime_state()
    if runtime_state:
        payload.update(runtime_state)
    payload.setdefault("vpn_target_index", 0)
    payload.setdefault("vpn_switch_history", [])
    payload.setdefault("last_vpn_result", None)
    payload.setdefault("last_public_ip", None)
    return payload


def limit_rows(rows: list[dict[str, str]]) -> list[dict[str, str]]:
    if MAX_ITEMS is None:
        return rows
    return rows[:MAX_ITEMS]


def load_initial_targets() -> tuple[list[dict[str, str]], list[dict[str, object]], list[dict[str, object]], dict[str, object], bool]:
    if CHECKPOINT_PATH.exists():
        payload = json.loads(CHECKPOINT_PATH.read_text(encoding="utf-8"))
        pending_rows = payload.get("pending_rows", [])
        for fallback_index, row in enumerate(pending_rows, start=1):
            row.setdefault("source_index", fallback_index)
        results = payload.get("results", [])
        failed_rows = payload.get("failed_rows", [])
        runtime_state = normalize_runtime_state(payload.get("runtime_state"))
        return pending_rows, results, failed_rows, runtime_state, True

    with CSV_PATH.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        all_rows = [
            {**row, "source_index": index}
            for index, row in enumerate(reader, start=1)
            if row.get("detail_url")
        ]
    return limit_rows(all_rows), [], [], build_default_runtime_state(), False


def write_snapshots(results: list[dict[str, object]], failed_rows: list[dict[str, object]]) -> None:
    RESULTS_JSON_PATH.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
    FAILED_JSON_PATH.write_text(json.dumps(failed_rows, ensure_ascii=False, indent=2), encoding="utf-8")
    write_csv_rows(RESULTS_CSV_PATH, results)
    write_csv_rows(FAILED_CSV_PATH, failed_rows)


def save_checkpoint(
    pending_rows: list[dict[str, str]],
    results: list[dict[str, object]],
    failed_rows: list[dict[str, object]],
    runtime_state: dict[str, object],
) -> None:
    payload = {
        "pending_rows": pending_rows,
        "results": results,
        "failed_rows": failed_rows,
        "runtime_state": runtime_state,
    }
    CHECKPOINT_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def clear_checkpoint() -> None:
    if CHECKPOINT_PATH.exists():
        CHECKPOINT_PATH.unlink()


def is_blocked_error(message: str | None) -> bool:
    if not message:
        return False
    markers = [
        "ERR_CONNECTION_CLOSED",
        "ERR_CONNECTION_RESET",
        "ERR_NETWORK_CHANGED",
        "ERR_HTTP2_PROTOCOL_ERROR",
    ]
    return any(marker in message for marker in markers)


def record_vpn_result(runtime_state: dict[str, object], switch_result: dict[str, object]) -> None:
    history = list(runtime_state.get("vpn_switch_history", []))
    history.append(switch_result)
    runtime_state["vpn_switch_history"] = history[-20:]
    runtime_state["last_vpn_result"] = switch_result
    runtime_state["vpn_target_index"] = switch_result.get("next_target_index", runtime_state.get("vpn_target_index", 0))
    if switch_result.get("current_ip"):
        runtime_state["last_public_ip"] = switch_result["current_ip"]


def build_vpn_target_sequence(runtime_state: dict[str, object]) -> list[str]:
    start_index = int(runtime_state.get("vpn_target_index", 0))
    return VPN_PRIMARY_TARGETS[start_index:] + VPN_PRIMARY_TARGETS[:start_index]


async def rotate_vpn_and_update_state(runtime_state: dict[str, object]) -> dict[str, object]:
    while True:
        start_index = int(runtime_state.get("vpn_target_index", 0))
        target_sequence = build_vpn_target_sequence(runtime_state)
        log(f"[VPN] 한국 서버 순환 시작 - start_index={start_index} targets={target_sequence}")

        try:
            switch_result = await vpn_switcher.rotate_vpn_connection(
                targets=VPN_PRIMARY_TARGETS,
                start_index=start_index,
                max_attempts=VPN_ROTATION_MAX_ATTEMPTS,
                connect_timeout_seconds=VPN_CONNECT_TIMEOUT_SECONDS,
                ready_timeout_seconds=VPN_READY_TIMEOUT_SECONDS,
                poll_interval_seconds=VPN_STATUS_POLL_INTERVAL_SECONDS,
                probe_target_url=VPN_PROBE_URL,
                probe_timeout_seconds=VPN_PROBE_TIMEOUT_SECONDS,
                max_probe_latency_seconds=VPN_MAX_PROBE_LATENCY_SECONDS,
            )
        except Exception as exc:  # noqa: BLE001
            switch_result = {
                "success": False,
                "target": None,
                "next_target_index": start_index,
                "current_ip": runtime_state.get("last_public_ip"),
                "error": str(exc),
                "elapsed_seconds": 0.0,
                "attempted_targets": target_sequence,
                "status_before": {},
                "status_after": None,
                "probe": None,
            }

        switch_result["target_sequence"] = target_sequence
        record_vpn_result(runtime_state, switch_result)

        if switch_result.get("success"):
            log(
                f"[VPN] 전환 성공 - target={switch_result.get('target')} next_target_index={switch_result.get('next_target_index')} ip={switch_result.get('current_ip')}"
            )
            return switch_result

        log(json.dumps(switch_result, ensure_ascii=False, indent=2))
        log(f"[VPN] 모든 한국 서버 실패. {int(VPN_RESET_WAIT_SECONDS)}초 대기 후 처음 서버부터 다시 시도합니다.")
        runtime_state["vpn_target_index"] = 0
        await asyncio.sleep(VPN_RESET_WAIT_SECONDS)


## 4. 실행 대상 로딩과 현재 상태 확인

- 입력 CSV 또는 기존 체크포인트에서 이번 실행 대상을 불러옵니다.
- 현재 남은 건수, 누적 성공/실패 건수, VPN 상태를 먼저 로그로 확인합니다.


In [ ]:
pending_rows, results, failed_rows, runtime_state, resumed_from_checkpoint = load_initial_targets()
write_snapshots(results, failed_rows)

log(f"체크포인트 재개 여부: {resumed_from_checkpoint}")
log(f"남은 대상 개수: {len(pending_rows)}")
log(f"기존 성공 개수: {len(results)}")
log(f"기존 실패 개수: {len(failed_rows)}")

try:
    current_vpn_status = await vpn_switcher.get_vpn_status()
    runtime_state["last_public_ip"] = runtime_state.get("last_public_ip") or current_vpn_status.ip
    log(json.dumps(current_vpn_status.to_dict(), ensure_ascii=False, indent=2))
except Exception as exc:  # noqa: BLE001
    log(f"VPN 상태 조회 실패: {exc}")

log(json.dumps(runtime_state, ensure_ascii=False, indent=2))
pending_rows[:2]


## 5. 상세페이지 배치 크롤링과 VPN 자동 전환

- 각 URL에 접속해 차량 상세 정보를 추출하고 `registration_number` 를 함께 저장합니다.
- 차단 의심 오류가 발생하면 체크포인트를 저장한 뒤 한국 VPN 서버만 순환합니다.
- 모든 한국 서버가 실패하면 3분 대기 후 처음 서버부터 다시 시도합니다.


In [ ]:
detail_semaphore = asyncio.Semaphore(DETAIL_CONCURRENCY)


async def process_one(row: dict[str, str]) -> dict[str, object]:
    detail_url = row["detail_url"]
    source_index = row["source_index"]
    demo_no = row.get("demo_no")
    item_started_at = time.perf_counter()
    detail_page = None
    log(f"[접속 시작] source_index={source_index} demo_no={demo_no} url={detail_url}")
    try:
        async with detail_semaphore:
            async with create_browser_context() as browser_context:
                detail_page = await crawl_detail_page(
                    detail_url,
                    timeout_ms=DETAIL_TIMEOUT_MS,
                    context=browser_context,
                )

        item_elapsed = time.perf_counter() - item_started_at
        registration_number = detail_page.registration_number if detail_page else None
        log(
            f"[접속 성공] source_index={source_index} demo_no={demo_no} registration_number={registration_number} elapsed={item_elapsed:.2f}s"
        )
        return {
            "status": "ok",
            "category": "ok",
            "order_index": source_index,
            "detail_url": detail_url,
            "demo_no": demo_no,
            "source_page": row.get("source_page"),
            "registration_number": registration_number,
            "detail_page": detail_page.to_dict() if detail_page else None,
            "elapsed_seconds": round(item_elapsed, 2),
            "error": None,
        }
    except BlockedRequestError as exc:
        category = "blocked"
        error_message = str(exc)
    except SlowResponseError as exc:
        category = "slow"
        error_message = str(exc)
    except StaleListingError as exc:
        category = "stale"
        error_message = str(exc)
    except Exception as exc:  # noqa: BLE001
        error_message = str(exc)
        category = "blocked" if is_blocked_error(error_message) else "error"

    item_elapsed = time.perf_counter() - item_started_at
    registration_number = detail_page.registration_number if detail_page else None
    log(
        f"[접속 오류] source_index={source_index} demo_no={demo_no} category={category} elapsed={item_elapsed:.2f}s error={error_message}"
    )
    return {
        "status": "error",
        "category": category,
        "order_index": source_index,
        "detail_url": detail_url,
        "demo_no": demo_no,
        "source_page": row.get("source_page"),
        "registration_number": registration_number,
        "detail_page": detail_page.to_dict() if detail_page else None,
        "elapsed_seconds": round(item_elapsed, 2),
        "error": error_message,
    }


attempt_total = 0
batch_number = 0
total_started_at = time.perf_counter()
chunk_started_at = total_started_at

while pending_rows:
    batch_number += 1
    current_batch = pending_rows[:BATCH_SIZE]
    remaining_rows = pending_rows[BATCH_SIZE:]
    log(f"[배치 {batch_number}] 시작 - 이번 배치 {len(current_batch)}건 / 전체 남은 대상 {len(pending_rows)}건")
    tasks = [asyncio.create_task(process_one(row)) for row in current_batch]

    batch_results = []
    blocked_rows = []
    retry_later_rows = []
    slow_rows = []

    for task in asyncio.as_completed(tasks):
        result = await task
        batch_results.append(result)
        attempt_total += 1
        log(
            f"[시도 {attempt_total}] 완료 - 차량번호: {result.get('registration_number')} / 소요: {result.get('elapsed_seconds')}초 / 오류: {result.get('error')}"
        )

        if attempt_total % LOG_EVERY == 0:
            chunk_elapsed = time.perf_counter() - chunk_started_at
            log(f"[로그] 최근 {LOG_EVERY}번 시도 시간: {chunk_elapsed:.2f}초")
            chunk_started_at = time.perf_counter()

    for row, result in zip(current_batch, sorted(batch_results, key=lambda item: item["order_index"])):
        if result["status"] == "ok":
            results.append(result)
            continue

        if result["category"] == "blocked":
            blocked_rows.append(row)
            continue

        if result["category"] == "slow":
            slow_retry_count = int(row.get("slow_retry_count", 0)) + 1
            slow_row = {**row, "slow_retry_count": slow_retry_count}
            if slow_retry_count >= SLOW_TO_BLOCK_THRESHOLD:
                blocked_rows.append(slow_row)
            elif slow_retry_count <= SLOW_RETRY_LIMIT:
                retry_later_rows.append(slow_row)
                slow_rows.append(slow_row)
                log(
                    f"[재시도 예정] source_index={row['source_index']} demo_no={row.get('demo_no')} slow_retry_count={slow_retry_count}"
                )
            else:
                failed_rows.append({**result, "slow_retry_count": slow_retry_count})
            continue

        failed_rows.append(result)

    if len(blocked_rows) == 0 and len(slow_rows) >= BATCH_SLOW_BLOCK_THRESHOLD:
        blocked_rows = slow_rows
        retry_later_rows = []
        log(f"느린 응답이 배치에서 {len(slow_rows)}건 발생하여 차단 의심으로 승격합니다.")

    pending_rows = blocked_rows + remaining_rows + retry_later_rows
    save_checkpoint(pending_rows, results, failed_rows, runtime_state)
    write_snapshots(results, failed_rows)

    if len(blocked_rows) >= BLOCKED_ERROR_THRESHOLD:
        log("차단 의심 오류 감지. 체크포인트 저장 후 VPN 자동 전환을 시도합니다.")
        switch_result = await rotate_vpn_and_update_state(runtime_state)
        save_checkpoint(pending_rows, results, failed_rows, runtime_state)
        write_snapshots(results, failed_rows)
        log(json.dumps(switch_result, ensure_ascii=False, indent=2))

        if not switch_result.get("success"):
            log("VPN 전환 실패. 체크포인트를 유지한 채 여기서 중단합니다.")
            break

        log("VPN 전환 완료. 같은 pending rows로 다시 진행합니다.")
        chunk_started_at = time.perf_counter()
        continue

    log(
        f"[배치 {batch_number}] 종료 - 성공 누적 {len(results)}건 / 실패 누적 {len(failed_rows)}건 / 재시도 예정 {len(retry_later_rows)}건 / 잔여 {len(pending_rows)}건"
    )

if not pending_rows:
    clear_checkpoint()

if attempt_total % LOG_EVERY != 0:
    chunk_size = attempt_total % LOG_EVERY
    chunk_elapsed = time.perf_counter() - chunk_started_at
    log(f"[로그] 최근 {chunk_size}번 시도 시간: {chunk_elapsed:.2f}초")

total_elapsed = time.perf_counter() - total_started_at
log(f"전체 소요 시간: {total_elapsed:.2f}초")
log(f"남은 대상 개수: {len(pending_rows)}")
log(f"성공 개수: {len(results)}")
log(f"실패 개수: {len(failed_rows)}")
log(f"결과 JSON 저장 경로: {RESULTS_JSON_PATH}")
log(f"실패 JSON 저장 경로: {FAILED_JSON_PATH}")
log(f"결과 CSV 저장 경로: {RESULTS_CSV_PATH}")
log(f"실패 CSV 저장 경로: {FAILED_CSV_PATH}")
log(f"로그 저장 경로: {LOG_PATH}")


## 6. 결과 미리보기와 저장 위치 확인

- 성공/실패 결과 일부와 현재 런타임 상태를 바로 확인합니다.
- 최종 CSV와 로그 파일 경로를 마지막으로 다시 보여줍니다.


In [ ]:
print(json.dumps(results[:3], ensure_ascii=False, indent=2))
print(json.dumps(failed_rows[:3], ensure_ascii=False, indent=2))
print(json.dumps(runtime_state, ensure_ascii=False, indent=2))
print({"pending": len(pending_rows), "results": len(results), "failed": len(failed_rows)})
print({"results_csv": str(RESULTS_CSV_PATH), "failed_csv": str(FAILED_CSV_PATH), "log_path": str(LOG_PATH)})
